## Langfuse Adapter Demo

This notebook demonstrates how to use the **HallucinationEvaluator** with the sklearn compatible **EPR** and **WEPR** detectors to score LLM traces in Langfuse.

### Prerequisites

* Install the adapter dependencies:
    ```bash
    uv pip install artefactual[adapters]
    ```

* Create a free LangFuse project at [cloud.langfuse.com](https://cloud.langfuse.com), then go to **Settings → API Keys**.

* Obtain an access token from your Hugging Face Account.

* API keys must be set as environment variables (e.g. via `direnv`):
    * `HF_TOKEN`, `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_HOST`

### Run a generation and send it to Langfuse

In [ ]:
import importlib.resources
import os
import time

from langfuse import get_client, observe
from langfuse.openai import OpenAI
from openai.types.chat import ChatCompletion

from artefactual.adapters.langfuse.evaluator import HallucinationEvaluator
from artefactual.scoring.base_detector import epr, wepr

# Calibrations shipped with the package. load_weights also accepts a registry
# model name, but only WEPR weights are registered, so address the files directly.
ARTEFACTS = importlib.resources.files("artefactual.data")

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)


@observe()
def run_generation() -> ChatCompletion:
    return client.chat.completions.create(
        model="Qwen/Qwen3-Coder-30B-A3B-Instruct",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is the capital of France?"},
        ],
        logprobs=True,
        top_logprobs=15,
    )


print("Generated message:", run_generation().choices[0].message.content)

langfuse = get_client()
langfuse.flush()

print("Waiting for Langfuse server to index the logprobs of the trace...")
time.sleep(3.0)

traces_to_evaluate = langfuse.api.trace.list(limit=1).data

### Score traces with EPR

A detector always needs calibration weights: pass a path to a weights file, or a registered model name. Calling `epr()` with no argument raises `UncalibratedModelError`.

In [ ]:
evaluator = HallucinationEvaluator(
    name="EPR",
    langfuse_client=langfuse,
    detector=epr(pretrained_model_name_or_path=str(ARTEFACTS / "calibration_ministral.json")),
)

for trace in traces_to_evaluate:
    score = evaluator.score_trace(trace.id)
    print(f"EPR Scored Trace : {trace.id} → {score}")

langfuse.flush()

### Score traces with WEPR

In [ ]:
evaluator = HallucinationEvaluator(
    name="WEPR",
    langfuse_client=langfuse,
    detector=wepr(pretrained_model_name_or_path=str(ARTEFACTS / "weights_ministral.json")),
)

for trace in traces_to_evaluate:
    score = evaluator.score_trace(trace.id)
    print(f"WEPR Scored Trace : {trace.id} → {score}")

langfuse.flush()